In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("../trump-vector/artifacts/congress_tweets.parquet")

In [3]:
df.head(30)

,id,link,screen_name,source,text,time,user_id,file_date
0,877527850420776961,https://www.twitter.com/CRN_Supplements/status...,RepErikPaulsen,Twitter Web Client,RT @CRN_Supplements Thank you @RepErikPaulsen ...,2017-06-21T10:05:17-04:00,17513304,2017-06-21
1,877628169028632576,https://www.twitter.com/RepTedBudd/statuses/87...,RepTedBudd,Twitter Web Client,Congrats to our Congressional Award Gold Medal...,2017-06-21T16:43:55-04:00,817138492614524928,2017-06-21
2,877580122785685504,https://www.twitter.com/SenatorWicker/statuses...,SenatorWicker,TweetDeck,ICYMI: I chaired a hearing to explore expandin...,2017-06-21T13:33:00-04:00,264219447,2017-06-21
3,877655119638048770,https://www.twitter.com/BernieSanders/statuses...,BernieSanders,TweetDeck,The Affordable Care Act should be improved. Bu...,2017-06-21T18:31:01-04:00,216776631,2017-06-21
4,877632313550553089,https://www.twitter.com/RepDonBacon/statuses/8...,RepDonBacon,Hootsuite,#TaxReform will create a low tax rate just for...,2017-06-21T17:00:23-04:00,818975124460335106,2017-06-21
5,877642732474310656,https://www.twitter.com/edworkforcedems/status...,RepLoisFrankel,Twitter Web Client,RT @edworkforcedems Your baby. Your family. Ou...,2017-06-21T17:41:47-04:00,1077121945,2017-06-21
6,877600342627635200,https://www.twitter.com/RepCloakroom/statuses/...,RepCloakroom,Twitter Web Client,We have begun 40 minutes of debate to suspend ...,2017-06-21T14:53:21-04:00,1137600571,2017-06-21
7,877636672116080640,https://www.twitter.com/dccc/statuses/87763667...,dccc,Buffer,Even though @BrianMastFL won #FL18 by 13.6 poi...,2017-06-21T17:17:42-04:00,14676022,2017-06-21
8,877653474992766976,https://www.twitter.com/SenJohnThune/statuses/...,SenJohnThune,TweetDeck,Great catching up with @SummitHoops COY @Coach...,2017-06-21T18:24:29-04:00,296361085,2017-06-21
9,877576142068228096,https://www.twitter.com/HughTFerguson/statuses...,RepGeneGreen,Twitter Web Client,RT @HughTFerguson Today on the Senate floor @S...,2017-06-21T13:17:11-04:00,111635527,2017-06-21


In [4]:
import re
import requests
from typing import Optional, List
import time

In [5]:
# Filter tweets with image URLs and legislator screen names, then sample 1000
handles_df = pd.read_csv("data/legislators_116_119_with_twitter.csv", usecols=["twitter"])
valid_handles = set(
    handles_df["twitter"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lstrip("@")
    .str.casefold()
)
valid_handles.discard("")

if "screen_name" not in df.columns:
    raise KeyError("Expected a 'screen_name' column in the tweets dataframe")

tweets_with_urls = df[df["text"].str.contains("pbs.twimg.com/media", case=False, na=False)].copy()
tweets_with_urls = tweets_with_urls[
    tweets_with_urls["screen_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lstrip("@")
    .str.casefold()
    .isin(valid_handles)
].copy()
print(f"Total tweets with URLs from matched legislator handles: {len(tweets_with_urls)}")

sample_size = min(1000, len(tweets_with_urls))
sampled_tweets = tweets_with_urls.sample(n=sample_size, random_state=42)
print(f"Sampled {len(sampled_tweets)} tweets")
sampled_tweets.head()

Total tweets with URLs from matched legislator handles: 672972
Sampled 1000 tweets


,id,link,screen_name,source,text,time,user_id,file_date
3880137,1486183402042802184,https://www.twitter.com/shahnazfarzaneh/status...,RepDonBacon,Twitter for iPhone,"RT @shahnazfarzaneh Thank you, @RepDonBacon. S...",2022-01-25T22:45:09-05:00,818975124460335106,2022-01-25
3167221,1372932793966194689,https://www.twitter.com/SenJohnBarrasso/status...,SenJohnBarrasso,Twitter for iPhone,Caroline Lockhart made huge strides for women ...,2021-03-19T11:27:40-04:00,202206694,2021-03-19
1592142,1155870169585111040,https://www.twitter.com/RepChrisPappas/statuse...,RepChrisPappas,Twitter for iPhone,Just finished a tour and met with employees at...,2019-07-29T11:58:18-04:00,1067748650485497862,2019-07-29
2144589,1240365572858220544,https://www.twitter.com/RepAdams/statuses/1240...,RepAdams,Twitter for iPhone,Thank you Senators for passing #FamiliesFirst!...,2020-03-18T15:52:52-04:00,2916086925,2020-03-18
145567,908505277682606080,https://www.twitter.com/RepEspaillat/statuses/...,RepEspaillat,Twitter for iPhone,Let's pass the #DreamAct and allow these young...,2017-09-14T21:38:32-04:00,817076257770835968,2017-09-14


In [6]:
sampled_tweets['text'].iloc[277]

"RT @ConorLambPA Second stop: Spaghetti Dinner at American Legion Post 760 in Bethel Park #PA18, where I applied to become a member &amp; got a new hat. Anyone who says politics isn't fun just isn't doing it right. http://pbs.twimg.com/media/DUlS-xyXUAAWMBK.jpg http://pbs.twimg.com/media/DUlTE6gWkAANKu9.jpg http://pbs.twimg.com/media/DUlTHLIWAAIZx4T.jpg"

## Wayback Machine API

In [48]:
import re
from typing import List, Optional, Dict, Any
import requests

WAYBACK_HEADERS = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "accept-language": "en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7",
    "cache-control": "max-age=0",
    "priority": "u=0, i",
    "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "document",
    "sec-fetch-mode": "navigate",
    "sec-fetch-site": "none",
    "sec-fetch-user": "?1",
    "upgrade-insecure-requests": "1",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
}


def get_wayback_cdx_snapshots(url: str) -> Optional[List[Dict[str, Any]]]:
    """
    Retrieve all matching snapshots for a URL from the Wayback CDX API.

    Args:
        url: The URL to search for in the Wayback Machine

    Returns:
        A list of snapshot dicts, or None if none found / request failed
    """
    try:
        cdx_api = "https://web.archive.org/cdx/search/cdx"
        params = {
            "url": url,
            "output": "json",
            "fl": "timestamp,original,statuscode,mimetype,digest,length",
            # collapse duplicate captures of the same content
            "filter": "statuscode:200",
            "limit": "50",
        }

        response = requests.get(
            cdx_api,
            params=params,
            headers=WAYBACK_HEADERS,
            timeout=15,
            allow_redirects=True,
        )
        response.raise_for_status()

        data = response.json()
        if not data or len(data) < 2:
            return None

        headers = data[0]
        rows = data[1:]

        snapshots = [dict(zip(headers, row)) for row in rows]
        return snapshots

    except Exception as e:
        print(f"Error fetching CDX data for {url}: {e}")
        return None


def build_wayback_url(timestamp: str, original_url: str) -> str:
    """
    Build a replay URL for a specific archived resource.
    """
    return f"https://web.archive.org/web/{timestamp}/{original_url}"


def extract_urls_from_tweet(text: str) -> List[str]:
    """Extract all twimg URLs from tweet text."""
    url_pattern = r'https?://pbs\.twimg\.com/[^\s]+'
    return re.findall(url_pattern, text)


# Test the function
test_url = "http://pbs.twimg.com/media/DDcnIPMXYAAaF0w.jpg"
print(f"Testing Wayback CDX API with {test_url}")

results = get_wayback_cdx_snapshots(test_url)

if results:
    print(f"Found {len(results)} snapshots")
    for snap in results[:5]:
        replay_url = build_wayback_url(snap["timestamp"], snap["original"])
        print({
            "timestamp": snap["timestamp"],
            "original": snap["original"],
            "statuscode": snap["statuscode"],
            "mimetype": snap["mimetype"],
            "replay_url": replay_url,
        })
else:
    print("No snapshots found")

Testing Wayback CDX API with http://pbs.twimg.com/media/DDcnIPMXYAAaF0w.jpg
Found 2 snapshots
{'timestamp': '20180929231010', 'original': 'https://pbs.twimg.com/media/DDcnIPMXYAAaF0w.jpg', 'statuscode': '200', 'mimetype': 'image/jpeg', 'replay_url': 'https://web.archive.org/web/20180929231010/https://pbs.twimg.com/media/DDcnIPMXYAAaF0w.jpg'}
{'timestamp': '20181029180536', 'original': 'https://pbs.twimg.com/media/DDcnIPMXYAAaF0w.jpg', 'statuscode': '200', 'mimetype': 'image/jpeg', 'replay_url': 'https://web.archive.org/web/20181029180536/https://pbs.twimg.com/media/DDcnIPMXYAAaF0w.jpg'}


In [ ]:
# Retrieve Wayback Machine snapshots for sampled tweets using CDX results
snapshot_rows = []

for idx, row in sampled_tweets.iterrows():
    tweet_urls = extract_urls_from_tweet(row["text"])

    for url in tweet_urls:
        # Small delay to avoid rate limiting
        time.sleep(5)
        snapshots = get_wayback_cdx_snapshots(url)
        print(f"Tweet ID {row.get('id', None)}: Found {len(snapshots) if snapshots else 0} snapshots for URL {url}")

        if snapshots:
            # Keep the most recent successful capture
            snapshot_info = snapshots[0]
            result_entry = {
                "tweet_id": row.get("id", None),
                "original_url": url,
                "wayback_timestamp": snapshot_info.get("timestamp"),
                "wayback_status": snapshot_info.get("statuscode"),
                "wayback_mimetype": snapshot_info.get("mimetype"),
                "wayback_url": build_wayback_url(snapshot_info.get("timestamp"), snapshot_info.get("original", url)),
            }
            snapshot_rows.append(result_entry)

# Create results dataframe
results_df = pd.DataFrame(snapshot_rows)
print(f"Retrieved {len(results_df)} snapshots from Wayback Machine")
if not results_df.empty:
    print(f"Status codes: {results_df['wayback_status'].value_counts().to_dict()}")
results_df.to_csv("data/wayback_snapshots.csv", index=False)
results_df.head(10)

In [ ]:
def download_image_from_wayback(wayback_url: str, timeout: int = 15) -> Optional[bytes]:
    """
    Download image content from a Wayback Machine URL.

    Args:
        wayback_url: The full Wayback Machine URL to an archived page
        timeout: Request timeout in seconds

    Returns:
        Image bytes if successful, None otherwise
    """
    try:
        response = requests.get(
            wayback_url,
            headers=WAYBACK_HEADERS,
            timeout=timeout,
            allow_redirects=True,
        )
        response.raise_for_status()
        return response.content
    except Exception as e:
        print(f"Error downloading from {wayback_url}: {e}")
        return None

# Try downloading sample images
print(f"Attempting to download {min(5, len(results_df))} sample archived pages...")
sample_results = results_df.head(5)

for idx, row in sample_results.iterrows():
    wayback_url = row['wayback_url']
    print(f"\nDownloading: {wayback_url}")
    image_bytes = download_image_from_wayback(wayback_url)

    if image_bytes:
        print(f"Retrieved {len(image_bytes)} bytes")
    else:
        print("Failed to retrieve content")

## Direct downloads

In [12]:
from pathlib import Path
from urllib.parse import urlparse
import hashlib
import re

# Output files for external wget workflow
URLS_TXT = Path("data/twitter_images.txt")
METADATA_CSV = Path("data/twitter_images_metadata.csv")
LEGISLATORS_CSV = Path("data/legislators_116_119_with_twitter.csv")

URLS_TXT.parent.mkdir(parents=True, exist_ok=True)


def _safe_ext_from_url(url: str) -> str:
    path = urlparse(url).path.lower()
    for ext in (".jpg", ".jpeg", ".png", ".gif", ".webp"):
        if path.endswith(ext):
            return ext
    return ".jpg"


def _build_filename(tweet_id, url: str) -> str:
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()[:12]
    ext = _safe_ext_from_url(url)
    return f"{tweet_id}_{digest}{ext}"


def _extract_direct_image_urls(text: str):
    url_pattern = r"https?://pbs\.twimg\.com/[^\s]+"
    urls = re.findall(url_pattern, text or "")
    return [u.rstrip('.,;:!?)\"\'') for u in urls if "pbs.twimg.com/media" in u]


# Build handle -> party mapping from legislators CSV
legislators_df = pd.read_csv(LEGISLATORS_CSV, usecols=["twitter", "party"])
legislators_df = legislators_df.dropna(subset=["twitter"]).copy()
legislators_df["twitter_norm"] = (
    legislators_df["twitter"].astype(str).str.strip().str.lstrip("@").str.casefold()
)
handle_to_party = (
    legislators_df.drop_duplicates(subset=["twitter_norm"])
    .set_index("twitter_norm")["party"]
    .to_dict()
)

rows = []
seen_tweet_url = set()
unique_urls = set()

for _, row in sampled_tweets.iterrows():
    tweet_id = row.get("id", "")
    screen_name = row.get("screen_name", "")
    screen_name_norm = str(screen_name).strip().lstrip("@").casefold()
    party = handle_to_party.get(screen_name_norm, "")
    text = row.get("text", "")

    urls = _extract_direct_image_urls(text)

    for url in urls:
        key = (tweet_id, url)
        if key in seen_tweet_url:
            continue
        seen_tweet_url.add(key)
        unique_urls.add(url)

        filename = _build_filename(tweet_id, url)
        rows.append(
            {
                "tweet_id": tweet_id,
                "screen_name": screen_name,
                "party": party,
                "url": url,
                "filename": filename,
                "relative_path": f"tweet_images/{filename}",
            }
        )

metadata_df = pd.DataFrame(rows)
metadata_df.to_csv(METADATA_CSV, index=False)

with URLS_TXT.open("w") as f:
    for url in sorted(unique_urls):
        f.write(url + "\n")

print(f"Wrote {len(unique_urls)} unique URLs to {URLS_TXT}")
print(f"Wrote {len(metadata_df)} tweet-url mappings to {METADATA_CSV}")
print("Use wget/curl externally to download URLs in twitter_images.txt")

metadata_df.head(10)

Wrote 1382 unique URLs to data/twitter_images.txt
Wrote 1383 tweet-url mappings to data/twitter_images_metadata.csv
Use wget/curl externally to download URLs in twitter_images.txt


,tweet_id,screen_name,party,url,filename,relative_path
0,1486183402042802184,RepDonBacon,Republican,http://pbs.twimg.com/media/FJ3_zIKWQAA2jni.jpg,1486183402042802184_9c84f8a21315.jpg,tweet_images/1486183402042802184_9c84f8a21315.jpg
1,1372932793966194689,SenJohnBarrasso,Republican,http://pbs.twimg.com/media/Ew2jGaoWQAEVUhh.jpg,1372932793966194689_58d7cef46ee2.jpg,tweet_images/1372932793966194689_58d7cef46ee2.jpg
2,1155870169585111040,RepChrisPappas,Democrat,http://pbs.twimg.com/media/EAp5bSiXsAAXXL4.jpg,1155870169585111040_cfa5a044b635.jpg,tweet_images/1155870169585111040_cfa5a044b635.jpg
3,1240365572858220544,RepAdams,Democrat,http://pbs.twimg.com/media/ETap6NaXQAEf0qE.jpg,1240365572858220544_08a5a82deafd.jpg,tweet_images/1240365572858220544_08a5a82deafd.jpg
4,908505277682606080,RepEspaillat,Democrat,http://pbs.twimg.com/media/DJuovBJWsAUUTCk.jpg,908505277682606080_7a85e6d3523a.jpg,tweet_images/908505277682606080_7a85e6d3523a.jpg
5,1112793800194363393,SenTedCruz,Republican,http://pbs.twimg.com/media/D3Fiqy3XcAwPaUd.jpg,1112793800194363393_9125e0c6cf08.jpg,tweet_images/1112793800194363393_9125e0c6cf08.jpg
6,1112793800194363393,SenTedCruz,Republican,http://pbs.twimg.com/media/D3Fiqy4XgAALtLP.jpg,1112793800194363393_57f93a1737ce.jpg,tweet_images/1112793800194363393_57f93a1737ce.jpg
7,1112793800194363393,SenTedCruz,Republican,http://pbs.twimg.com/media/D3Fiqy5XkAEwPZv.jpg,1112793800194363393_3b57ac9a2823.jpg,tweet_images/1112793800194363393_3b57ac9a2823.jpg
8,1112793800194363393,SenTedCruz,Republican,http://pbs.twimg.com/media/D3Fiqy5XQAAuDwD.jpg,1112793800194363393_19074658bcf0.jpg,tweet_images/1112793800194363393_19074658bcf0.jpg
9,1615854631132725252,ChrisVanHollen,Democrat,http://pbs.twimg.com/media/Fmyj5UQWAAgufok.jpg,1615854631132725252_971449238f58.jpg,tweet_images/1615854631132725252_971449238f58.jpg
